# Vamos a probar nuevos features para modelo de entrenamiento

In [13]:
import pandas as pd
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import os

## Establecer varaibles para lecturas de archivos

In [14]:
PATH = r"D:\Proyectos\Seminario\seminario-proyecto-grupo1\data\processed\clientes_features.csv"

## Lectura de archivo limpio y vamos a imprimir head

In [15]:
data_bruta = pd.read_csv(PATH)
data_bruta.head()

,invoice_number,product_code,product_description,product_quantity,invoice_date,product_price,customer_id,country
0,489434,85048,15cm christmas glass ball 20 lights,12,2009-12-01,6.95,13085,United Kingdom
1,489434,79323P,pink cherry lights,12,2009-12-01,6.75,13085,United Kingdom
2,489434,79323W,white cherry lights,12,2009-12-01,6.75,13085,United Kingdom
3,489434,22041,"record frame 7"" single size",48,2009-12-01,2.10,13085,United Kingdom
4,489434,21232,strawberry ceramic trinket box,24,2009-12-01,1.25,13085,United Kingdom


## Vamos a intentar calcular Recency
Eso significa el qué tan reciente los clientes hicieron una compra, agrupando la data por CustomerID y encontrando la última fecha de compra para cada cliente, luego calculando cuántos días han pasado desde la última compra

In [18]:
data_bruta['invoice_date'] = pd.to_datetime(data_bruta['invoice_date'])

In [19]:
db_recency = data_bruta.groupby(by='customer_id', as_index=False)['invoice_date'].max()
db_recency.columns = ['customer_id', 'last_purchase_date']
recent_date = db_recency['last_purchase_date'].max()
db_recency['recency'] = db_recency['last_purchase_date'].apply(lambda x: (recent_date - x).days)
db_recency.head()

,customer_id,last_purchase_date,recency
0,12346,2011-01-18,325
1,12347,2011-12-07,2
2,12348,2011-09-25,75
3,12349,2011-11-21,18
4,12350,2011-02-02,310


## Vamos a intentar calcular la Frecuency
Esto es qué tan a menudo un cliente realiza una compra debemos intentar el quitar duplicados, vamos a ver cómo lo hace exactamente el método drop.duplicates

In [20]:
db_frecuency = data_bruta.drop_duplicates(subset=['invoice_number', 'product_code', 'customer_id']).groupby(by='customer_id', as_index=False)['invoice_date'].count()
db_frecuency.columns = ['customer_id', 'frequency']
db_frecuency.head()

,customer_id,frequency
0,12346,34
1,12347,222
2,12348,47
3,12349,175
4,12350,17


## Vamos a intentar calcular el valor Monetario
Esto es cuánto gasta un cliente en las compras sumamos el valor monetario gastado por cada cliente para obtener el total gastado

In [22]:
data_bruta['TotalSum'] = data_bruta['product_price'] * data_bruta['product_quantity']
db_monetary = data_bruta.groupby(by="customer_id", as_index=False)['TotalSum'].sum()
db_monetary.columns = ['customer_id', 'monetary']
db_monetary.head()

,customer_id,monetary
0,12346,77556.46
1,12347,5633.32
2,12348,2019.40
3,12349,4428.69
4,12350,334.40


## Mergear recency, frecuency, y monetary data
Vamos a unir los 3 indicadores por cada cliente en un dataFrame esto nos dara una vista comprensiva del comportamiento de cada cliente

In [23]:
df_rf = db_recency.merge(db_frecuency, on = 'customer_id')
df_rfm = df_rf.merge(db_monetary, on = 'customer_id').drop(columns=['last_purchase_date'])
df_rfm.head()

,customer_id,recency,frequency,monetary
0,12346,325,34,77556.46
1,12347,2,222,5633.32
2,12348,75,47,2019.40
3,12349,18,175,4428.69
4,12350,310,17,334.40


## Rankeando customers basados en los 3 indicadores
Ranking basado en recency, frecuency y monetary. El más bajo recency es mejor y las más altas frecuencias y valores monetarios son mejor, se asigna el rankeo a cada cliente.

In [24]:
df_rfm['R_rank'] = df_rfm['recency'].rank(ascending=False)
df_rfm['F_rank'] = df_rfm['frequency'].rank(ascending=True)
df_rfm['M_rank'] = df_rfm['monetary'].rank(ascending=True)

df_rfm.head()

,customer_id,recency,frequency,monetary,R_rank,F_rank,M_rank
0,12346,325,34,77556.46,1702.5,2236.5,5860.0
1,12347,2,222,5633.32,5695.5,4989.5,5300.0
2,12348,75,47,2019.40,3186.0,2782.5,4232.0
3,12349,18,175,4428.69,4749.5,4735.5,5112.0
4,12350,310,17,334.40,1767.5,1238.0,1409.0


## Normalizando los rankings
Se deben normalizar los rankings en una escala de 0-100 para hacerlos fáciles de comparar, el rankeo se hace consistente de acuerdo a los diferentes cliente y ayuda a calcular el RFM final

In [25]:
df_rfm['R_rank_norm'] = (df_rfm['R_rank'] / df_rfm['R_rank'].max()) * 100
df_rfm['F_rank_norm'] = (df_rfm['F_rank'] / df_rfm['F_rank'].max()) * 100
df_rfm['M_rank_norm'] = (df_rfm['M_rank'] / df_rfm['M_rank'].max()) * 100
df_rfm.head()

,customer_id,recency,frequency,monetary,R_rank,F_rank,M_rank,R_rank_norm,F_rank_norm,M_rank_norm
0,12346,325,34,77556.46,1702.5,2236.5,5860.0,29.047944,38.048656,99.693773
1,12347,2,222,5633.32,5695.5,4989.5,5300.0,97.176250,84.884314,90.166723
2,12348,75,47,2019.40,3186.0,2782.5,4232.0,54.359324,47.337530,71.997278
3,12349,18,175,4428.69,4749.5,4735.5,5112.0,81.035659,80.563117,86.968357
4,12350,310,17,334.40,1767.5,1238.0,1409.0,30.156970,21.061586,23.970738


## Vamos a sacar los rankings de los indicadores individuales
Eso es para limpiar el data frame y limpia la data

In [26]:
df_rfm.drop(columns=['R_rank', 'F_rank', 'M_rank'], inplace=True)
df_rfm.head()

,customer_id,recency,frequency,monetary,R_rank_norm,F_rank_norm,M_rank_norm
0,12346,325,34,77556.46,29.047944,38.048656,99.693773
1,12347,2,222,5633.32,97.176250,84.884314,90.166723
2,12348,75,47,2019.40,54.359324,47.337530,71.997278
3,12349,18,175,4428.69,81.035659,80.563117,86.968357
4,12350,310,17,334.40,30.156970,21.061586,23.970738


## Ahora vamos a calcular el RFM Score
El score RFM se calcula asignando pesos de importancia a los diferentes indicadores (Recency, Frecuency, Monetary), estos pesos se dan de acuerdo a la importancia y reglas del negocio, el monetario siempre será el más alto

In [29]:
# Asignación de pesos ponderados y cálculo del RFM Score
df_rfm['RFM_Score'] = 0.15 * df_rfm['R_rank_norm'] + 0.28 * df_rfm['F_rank_norm'] + 0.57 * df_rfm['M_rank_norm']
# Asignación de escala de 0 a 5 para reducir el rango del RFM Score y su mejora en la segmentación
df_rfm['RFM_Score'] *= 0.05
df_rfm.head()

,customer_id,recency,frequency,monetary,R_rank_norm,F_rank_norm,M_rank_norm,RFM_Score
0,12346,325,34,77556.46,29.047944,38.048656,99.693773,3.591813
1,12347,2,222,5633.32,97.176250,84.884314,90.166723,4.486954
2,12348,75,47,2019.40,54.359324,47.337530,71.997278,3.122343
3,12349,18,175,4428.69,81.035659,80.563117,86.968357,4.214249
4,12350,310,17,334.40,30.156970,21.061586,23.970738,1.204206
